# Transformer Day Exercises

In [1]:
# Set Up
#!git clone https://github.com/LxMLS/lxmls-toolkit.git
#%cd lxmls-toolkit/
import sys
import os
sys.path.append("../../../")

## Exercise 1: Tokenization

*Tokenization* is a fundamental process in modern NLP pipelines. It works by splitting a sentence, which consists in a sequence of characters, into atomic elements called **tokens**, possibly discarding other elements like punctuations. These tokens are the basic language units used by the model. 

As we will see, the granularity of such units may cary. 
In general, *tokenization allows us to represent text data in a format that can be process by standar deep learning models*. In this exercise, we will explore tokenization to understand how it helps in NLP tasks.

### Word-based tokenizers

The easiest way to split a text into units is probably to divide it based on whitespace. The idea is to chunk a sentence into segments everytime a space charecter is found. In python, we can do this with the `split()` function.

_string.split(separator, maxsplit)_

where separator is the whitespace by default and there's not maxsplit. You can read the python docs for `split()` [here](https://docs.python.org/3/library/stdtypes.html#str.split)

In [2]:
text = "I travelled to Lisbon in July to attend an NLP summer school"
text.split()

['I',
 'travelled',
 'to',
 'Lisbon',
 'in',
 'July',
 'to',
 'attend',
 'an',
 'NLP',
 'summer',
 'school']

Now, this list of words can be fed to a model to perform task like next token prediction, machine translation, story generation etc.

Tokenizing based on words allows the model to understand the basic units of language withouth worrying to learn things like workd boundaries. However, a downside of this approach is that we end up having an extremely large vocabulary, with an entry for each word of the language.
Due to computational and processing resources, deep learning models still struggle to handle vocabularies that are larger than tens of thousands of tokens. For this reasons, some words need to be left out of the vocabulary and are all mapped to a shared token usually called UNK, which stands for unknown (word).

#### Text normalization

In actual tokenization pipelines, text is usually normalized before being tokenized. This process means extracting a basic version of each word that is stripped from suffixes or functional information. For example, the verb "run" might appear as "running," "runs" or "ran", and the word "program" might appear as "programmer".


### Character-based tokenizers

Another option is to tokenize text based on **characters**. This allows us to have a much smaller model vocabulary, it provides us with a method to handle new words given the combinatorial ability of combining known characters, but it doesn't instill to the model the concept of words, which should be learned during training.

In [3]:
text = "I will travel to Lisbon in July, I will attend an NLP summer school but I hope to visit around: it's a beatiful city!"
tokenized = [c for c in text if c not in [",", ";", ":", "'", "!", "?"]]
print(tokenized)

['I', ' ', 'w', 'i', 'l', 'l', ' ', 't', 'r', 'a', 'v', 'e', 'l', ' ', 't', 'o', ' ', 'L', 'i', 's', 'b', 'o', 'n', ' ', 'i', 'n', ' ', 'J', 'u', 'l', 'y', ' ', 'I', ' ', 'w', 'i', 'l', 'l', ' ', 'a', 't', 't', 'e', 'n', 'd', ' ', 'a', 'n', ' ', 'N', 'L', 'P', ' ', 's', 'u', 'm', 'm', 'e', 'r', ' ', 's', 'c', 'h', 'o', 'o', 'l', ' ', 'b', 'u', 't', ' ', 'I', ' ', 'h', 'o', 'p', 'e', ' ', 't', 'o', ' ', 'v', 'i', 's', 'i', 't', ' ', 'a', 'r', 'o', 'u', 'n', 'd', ' ', 'i', 't', 's', ' ', 'a', ' ', 'b', 'e', 'a', 't', 'i', 'f', 'u', 'l', ' ', 'c', 'i', 't', 'y']


**Question** What other problem are character-based tokenizers posing to NLP models?

**Your Answer**: 

### The best of both worlds: subword-based tokenizers

In order to combine the best of both words, the most frequent tokenization strategy for modern NLP system is to tokenize based on **subwords**. Subwords are sequence of characters the can be shorter than entire words. 

How to split words into subwords depends on *how frequent a given sequence of characters is*. The core idea is that very frequent sequences are not split given that they are very likely to be used and appear a lot in corpora.

For instance, "unexpectedly" can be a **rare word** in a corpus and being split in the subwords "un", "expected", "ly".
These are standalone subwords that can be reused across other words, while the meaning of "unexpectedly" can be retained combining the three subwords. On the other hand, words like cats, people, and running, given their high frequency, will not be split by the tokenizer.


#### BPE (byte-pair encoding) tokenizer

One of the most widely used subword tokenizer is based on a method called **BPE (byte-pair encoding)**, which was introduced in a paper by [Senrich et al, 2016](https://aclanthology.org/P16-1162/). BPE relies on an *iterative algorithm based on the frequency of character sequences*. At each step, the algorithm compute character frequencies and merges pairs that tend to occur together

##### Notes

Note that after we split words into subwords, we are now left with all the elements that will form the *vocabulary of our model*. Each subword is then mapped into an **index in the model vocabulary**. This is a standard mapping between string respresentations of (sub)words and vocabulary entries. 

Remember that subword tokenizers need to be "trained", they need to learn what word splits are based on data that they see. This is not the same training that we do with our neural network models given that tokenizers are *non-parametric deterministic methods*. 

Lastly, note also that different data leads to different word splitting choices and that a tokenizer is directly connected to a model. For this reason, you have keep in mind that a model trained on a given tokenizer it is not guaranteed to perform equally when coupled with a different tokenizer.

##### Extra:

Each vocabulary index, which corresponds to a subword, is then used by the model to load and process a related (sub)word embedding representation. These are dense vectors that the model will use when doing computation for any NLP task. If you want to learn more about word embeddings, check out this [website](https://lena-voita.github.io/nlp_course/word_embeddings.html).

### Using BPE

We are now looking at a real example using BPE. We can import the BPE tokenizer from the lxmls toolkit. BPE, which is used in models like GPT-2, and other commonly used subword tokenizers like WordPiece (used in BERT) are available in practically any standard NLP library like huggingface.

In [4]:
from lxmls.transformers.bpe import BPETokenizer

In [5]:
tokenizer = BPETokenizer()

We will now split the sentence: *"Your drawing is charmingly anachronistic."*

**Question:** Do you have a guess on which word will be split subwords and which one won't?

In [6]:
# Tokenize a sample sentence
sentence = "Your drawing is charmingly anachronistic."
tokenizer.encoder.encode_and_show_work(sentence)

{'bpe_idx': [7120, 8263, 318, 23332, 306, 281, 620, 1313, 2569, 13],
 'tokens': ['Your', ' drawing', ' is', ' charmingly', ' anachronistic', '.'],
 'parts': [{'token': 'Your',
   'token_bytes': b'Your',
   'token_translated': 'Your',
   'token_merged': ['Your'],
   'token_ix': [7120]},
  {'token': ' drawing',
   'token_bytes': b' drawing',
   'token_translated': 'Ġdrawing',
   'token_merged': ['Ġdrawing'],
   'token_ix': [8263]},
  {'token': ' is',
   'token_bytes': b' is',
   'token_translated': 'Ġis',
   'token_merged': ['Ġis'],
   'token_ix': [318]},
  {'token': ' charmingly',
   'token_bytes': b' charmingly',
   'token_translated': 'Ġcharmingly',
   'token_merged': ['Ġcharming', 'ly'],
   'token_ix': [23332, 306]},
  {'token': ' anachronistic',
   'token_bytes': b' anachronistic',
   'token_translated': 'Ġanachronistic',
   'token_merged': ['Ġan', 'ach', 'ron', 'istic'],
   'token_ix': [281, 620, 1313, 2569]},
  {'token': '.',
   'token_bytes': b'.',
   'token_translated': '.',
   

We can now look at this python vocabulary object returned by the tokenizer. If you look at `bpe_idx`, you see all the subwords that the tokenizer decided to split. These number are the indexes in the vocabulary all our model. The `parts` field contains the actual splittting in the `token_merged` field. 

As you can see, some words have been split. Do they match your initial guess? Why?

#### Whitespaces

You probably have noticed a special `Ġ` character inserted before each word expect the first one. This is because spaces are converted into this special token by the BPE algorithm, such that the word "run" and " run" are not treated equally and the tokenizer understands whether a word is at the beginning of a sentence or not. 

This is something that was found to provide better performance to the original GPT2 model. For more information you can read [here](https://discuss.huggingface.co/t/bpe-tokenizers-and-spaces-before-words/475?u=joaogante)

Now look at the next two tokenized sentence. Can you notice how the word "very" is assigned two different tokens?

In [7]:
sentence = "running is very cool"
tokenizer.encoder.encode_and_show_work(sentence)

{'bpe_idx': [20270, 318, 845, 3608],
 'tokens': ['running', ' is', ' very', ' cool'],
 'parts': [{'token': 'running',
   'token_bytes': b'running',
   'token_translated': 'running',
   'token_merged': ['running'],
   'token_ix': [20270]},
  {'token': ' is',
   'token_bytes': b' is',
   'token_translated': 'Ġis',
   'token_merged': ['Ġis'],
   'token_ix': [318]},
  {'token': ' very',
   'token_bytes': b' very',
   'token_translated': 'Ġvery',
   'token_merged': ['Ġvery'],
   'token_ix': [845]},
  {'token': ' cool',
   'token_bytes': b' cool',
   'token_translated': 'Ġcool',
   'token_merged': ['Ġcool'],
   'token_ix': [3608]}]}

In [8]:
sentence = "very cool!"
tokenizer.encoder.encode_and_show_work(sentence)

{'bpe_idx': [548, 3608, 0],
 'tokens': ['very', ' cool', '!'],
 'parts': [{'token': 'very',
   'token_bytes': b'very',
   'token_translated': 'very',
   'token_merged': ['very'],
   'token_ix': [548]},
  {'token': ' cool',
   'token_bytes': b' cool',
   'token_translated': 'Ġcool',
   'token_merged': ['Ġcool'],
   'token_ix': [3608]},
  {'token': '!',
   'token_bytes': b'!',
   'token_translated': '!',
   'token_merged': ['!'],
   'token_ix': [0]}]}

#### Handling typos

Now we are going to tokenize another two very similar sentences.

In [9]:
sentence = "I like to circumnavigate the globe every year"
tokenizer.encoder.encode_and_show_work(sentence)

{'bpe_idx': [40, 588, 284, 2498, 4182, 615, 10055, 262, 13342, 790, 614],
 'tokens': ['I',
  ' like',
  ' to',
  ' circumnavigate',
  ' the',
  ' globe',
  ' every',
  ' year'],
 'parts': [{'token': 'I',
   'token_bytes': b'I',
   'token_translated': 'I',
   'token_merged': ['I'],
   'token_ix': [40]},
  {'token': ' like',
   'token_bytes': b' like',
   'token_translated': 'Ġlike',
   'token_merged': ['Ġlike'],
   'token_ix': [588]},
  {'token': ' to',
   'token_bytes': b' to',
   'token_translated': 'Ġto',
   'token_merged': ['Ġto'],
   'token_ix': [284]},
  {'token': ' circumnavigate',
   'token_bytes': b' circumnavigate',
   'token_translated': 'Ġcircumnavigate',
   'token_merged': ['Ġcirc', 'umn', 'av', 'igate'],
   'token_ix': [2498, 4182, 615, 10055]},
  {'token': ' the',
   'token_bytes': b' the',
   'token_translated': 'Ġthe',
   'token_merged': ['Ġthe'],
   'token_ix': [262]},
  {'token': ' globe',
   'token_bytes': b' globe',
   'token_translated': 'Ġglobe',
   'token_merged'

In [10]:
sentence = "I like to cirkumnavigate the globe every year"
tokenizer.encoder.encode_and_show_work(sentence)

{'bpe_idx': [40, 588, 284, 10774, 74, 4182, 615, 10055, 262, 13342, 790, 614],
 'tokens': ['I',
  ' like',
  ' to',
  ' cirkumnavigate',
  ' the',
  ' globe',
  ' every',
  ' year'],
 'parts': [{'token': 'I',
   'token_bytes': b'I',
   'token_translated': 'I',
   'token_merged': ['I'],
   'token_ix': [40]},
  {'token': ' like',
   'token_bytes': b' like',
   'token_translated': 'Ġlike',
   'token_merged': ['Ġlike'],
   'token_ix': [588]},
  {'token': ' to',
   'token_bytes': b' to',
   'token_translated': 'Ġto',
   'token_merged': ['Ġto'],
   'token_ix': [284]},
  {'token': ' cirkumnavigate',
   'token_bytes': b' cirkumnavigate',
   'token_translated': 'Ġcirkumnavigate',
   'token_merged': ['Ġcir', 'k', 'umn', 'av', 'igate'],
   'token_ix': [10774, 74, 4182, 615, 10055]},
  {'token': ' the',
   'token_bytes': b' the',
   'token_translated': 'Ġthe',
   'token_merged': ['Ġthe'],
   'token_ix': [262]},
  {'token': ' globe',
   'token_bytes': b' globe',
   'token_translated': 'Ġglobe',
   

Why have they been tokenized differently? 

The only difference between these two sentences is in the the **typo** of the word "circumnavigate". As you can see, a _simple change_ in the word spelling breaks the tokenization process and leads to a different result. However, unlike word-based tokenizers where the wrong word would have been processed as an _unknown_ word, here we can still retain some ther other correct characters and our favorite NLP model can hopefully partially make it up for the typo while processing the sentence.

#### Determinism

Finally, recall that another important aspect of the tokenization process is that it's fully deterministic. Once we split a sentence into chunks and obtain the list of word indexes, we can fully revert the process and decode back the original text.

In [11]:
original_sentence = "We are about to start exercise 2 about attention, let's have fun!"
tokenized_sentence = tokenizer.encoder.encode(original_sentence)
reconstructed_sentence = tokenizer.encoder.decode(tokenized_sentence)

print(reconstructed_sentence)

We are about to start exercise 2 about attention, let's have fun!


## Excercise 2: Attention

Attention is a crucial component in the transformer, it allows to capture dependencies between different positions of two sequence of elements. In our case, and in most cases in NLP applications, sequences are sentences and elements are (sub)words.
It is a powerful operation that allows to learn an alignment between each element in two sequences. It generates a score of how related each element in sequence1 and sequence2 are between each other.
Understanding how attention works and being able to implement it are essential for anyone working with transformers. 

Given a query ($Q$), key ($K$), and value ($V$) tensors, the attention mechanism computes a weighted sum of the value tensor based on the similarity between the query and key tensors as shown in the following equation:

$$
\text{Attention}(Q,K,V) = \text{softmax}\Big(\frac{QK^T}{\sqrt{d_k}}\Big)V
$$

where 
- $Q$ represents the query tensor.
- $K$ represents the key tensor.
- $V$ represents the value tensor.
- $d_k$ represents the dimensionality of the key tensor.

This is the image that was in the original Transformer paper and that shows the computations used in the attention.

Forget about the right part, we'll get back to that later in the lab.

![image](https://miro.medium.com/v2/resize:fit:1270/1*LpDpZojgoKTPBBt8wdC4nQ.png)


In this exercise, we will dive into the attention mechanism. To do so, we are going to build a simple cross-attention function that we will then extend to a more complex multi-head self-attention module that incorporates the concept of causality.

### Exercise 2.1: Building a Simple Cross-Attention Function

Cross-attention refers to the case where the input sequences to compute $Q$, $K$, and $V$ come from different sources. It allows models to incorporate contextual information from one sequence (S1) into another (S2). <a name="cite_ref-1"></a>[<sup>[1]</sup>](#cite_note-1)


Given two input sequences $S_1$ and $S_2$ and the transformation weights $W_Q$, $W_K$ and $W_V$, complete the `cross_attention` function in the cell below. 

You need to implement the following:
- Calculate the query, key, and value projections using linear transformations.
- Compute the attention scores by performing the dot product between the query and key tensors.
- Apply softmax activation to the attention scores to obtain the attention weights.
- Multiply the attention weights with the value tensor to get the attended values.
- Return the attended values.

<a name="cite_note-1"></a>1. [^](#cite_ref-1) Conceptually, the self attention variant that you might have heard is the same, with the only difference that the S1 and S2 sequences are the same.

Hint: Matrix sizes

- q: query size
- d: hidden dimension
- c: context length
---
- Q: 1xqxd
- K, V: 1xcxd
- Q x K: 1xqxd x 1xsxd.T
- (QK) x V: 1xqxs x V 1xsxd
---
- Attn: 1xqxd

In [19]:
import torch
import torch.nn.functional as F

def cross_attention(S1, S2, W_Q, W_K, W_V):
    
    # Calculate Queries from sequence S2
    queries = S2 @W_Q
    # Calculate Key and Value from sequence S1
    keys = S1 @ W_K
    values = S1 @ W_V
    # Compute attention scores
    attention = queries@ keys.transpose(1,2)
    # Scale the attention scores
    attention /= (W_K.shape[1]**0.5)
    # Apply softmax to obtain attention weights
    attention = F.softmax(attention, dim=-1)
    # Compute the attended values
    attended_values = attention@values
    return attended_values

In [20]:
# Context
S1 = torch.rand((1,13,3))

# Query
S2 = torch.rand((1,4,3))

# Projections
W_Q = torch.rand((3, 2))  # Query weights
W_K = torch.rand((3, 2))  # Key weights
W_V = torch.rand((3, 2))  # Value weights

# Perform cross-attention
attended_values = cross_attention(S1, S2, W_Q, W_K, W_V)

# Expected output # 1,4,2 (B, Sequence, Projection)
print(f"Output Shape: {attended_values.shape}")

Output Shape: torch.Size([1, 4, 2])


In [21]:
%debug

> /tmp/ipykernel_26716/4102274962.py(12)cross_attention()
     10     values = S1 @ W_V
     11     # Compute attention scores
---> 12     attention = queries@ keys.T
     13     # Scale the attention scores
     14     attention /= (W_K.shape[1]**0.5)



ipdb>  q


### Exercise 2.2: Extending to Multi-Head Self-Attention
Great! You have successfully implemented cross-attention. Now, let's make some modifications so we can train a real GPT model.

**1. Self-Attention**

We will be replacing the cross-attention mechanism with self-attention. In self-attention, a single sequence acts as the query, key, and value, allowing attention to be computed within the sequence itself. This can be useful for syntactic where an attention head can model the relationship between part of speech like subjects and verbs. 

**2. Multi-Head**

However, the relations present even in a single sentence are more than one. Think about number and gender agreement as one, the semantic relation between subject and object, the functional aspect that verb arguments have etc. All this cannot be modeled by a single head.

For this reason, we are going to extend the single-head attention function to **multi-head attention**. In the previous implementation, we had one set of weights for the input query, resulting in a single type of _relationship between the the source and target sequence_. With multi-head attention, we can utilize _multiple parallel single-head attention modules_ to obtain diverse relationships between the query and the values. The attention operation works by projecting the sequences through a multiplication with a projection matrix, and then computing the alignment score. These are are all operation that can be parallelized since there's no interdependency between each each head. For this reasons, each head could learn to model a different linguistic intereation useful for many downstream tasks, be it syntactic, semantic or generation-based..

**3. Pytorch Module**

The last modification involves embedding our function into a PyTorch module. As you may have noticed, in the previous exercise, we passed the transformation weights as inputs to the function. In a real-world scenario, these matrices are learned, and PyTorch can keep track of them for us.

Complete the missing lines on the initialization of the module and the forward pass.

###### Note

GPT uses a version of self-attention called causal self-attention. When training our models for tasks like language modeling and machine translation, in practice we feed the entire train sequence to the model but, at every timestep, we want to prevent it to compute the alignment with future tokens. For this reason we use a mask that we incrementally lift at every timestep. For instance, we have a sentence that says "Libson is a great city to live in". At time 0, we feed the entire sentence to the model masking everything but the first token. Using the strikethrough format as masking, this will be what the model sees at step 0:

- Time 0: Libson ~is a great city to live in~

We then let the model generate a token a and move to step 1 where we are masking everything but the first two tokens
 
- Time 1: Libson is ~a great city to live in~ 

and so on...

- Time 2: Libson is a ~great city to live in~ 
- Time 3: Libson is a great ~city to live in~ 
- Time 4: Libson is a great city ~to live in~ 
- Time 5: Libson is a great city to ~live in~ 
- Time 6: Libson is a great city to live ~in~ 

We can now look back at the attention figure from the paper. Hopefully, you are now able to understand also the right side of the figure.

![image](https://miro.medium.com/v2/resize:fit:1270/1*LpDpZojgoKTPBBt8wdC4nQ.png)

In [3]:
"""
Full definition of a GPT Language Model, all of it in this single file.

References:
1) the official GPT-2 TensorFlow implementation released by OpenAI:
https://github.com/openai/gpt-2/blob/master/src/model.py
2) huggingface/transformers PyTorch implementation:
https://github.com/huggingface/transformers/blob/main/src/transformers/models/gpt2/modeling_gpt2.py
"""

import math
import re
import torch
import torch.nn as nn
from torch.nn import functional as F

from lxmls.transformers.utils import CfgNode as CN
from lxmls.transformers.bpe import BPETokenizer
from lxmls.transformers.pretrained_attention import PretrainedCausalSelfAttention

# -----------------------------------------------------------------------------


class NewGELU(nn.Module):
    """
    Implementation of the GELU activation function currently in Google BERT repo (identical to OpenAI GPT).
    Reference: Gaussian Error Linear Units (GELU) paper: https://arxiv.org/abs/1606.08415
    """

    def forward(self, x):
        return 0.5 * x * (1.0 + torch.tanh(
            math.sqrt(2.0 / math.pi) * (x + 0.044715 * torch.pow(x, 3.0))))


class CausalSelfAttention(nn.Module):
    """
    A vanilla multi-head masked self-attention layer with a projection at the end.
    It is possible to use torch.nn.MultiheadAttention here but I am including an
    explicit implementation here to show that there is nothing too scary here.
    """

    def __init__(self, config):
        super().__init__()

        # Initialize layers and parameters
        self.hidden_size = config.n_embd
        self.num_heads = config.n_head

        # Create the linear projections for query, key, and value tensors
        # Note: the input and output size of all these projections is n_embd
        self.query_proj = nn.Linear(config.n_embd, config.n_embd)
        self.key_proj = nn.Linear(config.n_embd, config.n_embd)
        self.value_proj = nn.Linear(config.n_embd, config.n_embd)

        self.output_proj = nn.Linear(config.n_embd, config.n_embd)

        self.attn_dropout = nn.Dropout(config.attn_pdrop)
        self.resid_dropout = nn.Dropout(config.resid_pdrop)

        self.register_buffer(
            "bias",
            torch.tril(torch.ones(config.block_size, config.block_size)).view(
                1, 1, config.block_size, config.block_size))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.size()

        # ----------
        # Solution to Exercise 2.2.1

        # Create the projections for query, key, and value tensors
        # Note: In self-attention these are all over the same tensor x

        query = self.query_proj(x)
        key = self.key_proj(x)
        value = self.value_proj(x)

        # End solution to Exercise 2.2.1
        # ----------

        # Reshape and transpose tensors for multi-head computation.
        # We reshape the output from (B, T, C) to (B, T, num_heads, hidden_size/num_heads)
        # And transpose the result to (B, num_heads, T, hidden_size/num_heads)
        # So the multi-head computation can be implemented as a single matrix multiplication.
        query = query.view(B, T, self.num_heads,
                           self.hidden_size // self.num_heads).transpose(1, 2)
        key = key.view(B, T, self.num_heads,
                       self.hidden_size // self.num_heads).transpose(1, 2)
        value = value.view(B, T, self.num_heads,
                           self.hidden_size // self.num_heads).transpose(1, 2)

        # ----------
        # Solution to Exercise 2.2.2

        # Compute attention scores. The shape of scores should be (B, num_heads, T, T)
        # Hint: You can use tensor.transpose() to adapt the order of the axes.

        # Normalize the scores by dividing by the square root of the hidden size
        # Take into account that you are using multi-head attention!

        scores = query@ key.transpose(2,3)
        scores = scores / (self.hidden_size // self.num_heads)**0.5
        # End solution to Exercise 2.2.2
        # ----------

        # Apply causal mask to restrict attention to the left in the input sequence
        mask = self.bias[:, :, :T, :T]
        scores = scores.masked_fill(mask == 0, float('-inf'))

        # ----------
        # Solution to Exercise 2.2.3

        # Apply softmax activation to get attention weights
        # Check the correct axis for the softmax function! What should be the shape of the weights?

        weights = F.softmax(scores, dim=-1)

        # End solution to Exercise 2.2.3
        # ----------

        # Apply dropout to the attention weights
        weights = self.attn_dropout(weights)

        # ----------
        # Solution to Exercise 2.2.4

        # Multiply attention weights with values to get attended values

        attended_values = weights@value

        # End solution to Exercise 2.2.4
        # ----------

        # Transpose and reshape attended values to restore original shape
        attended_values = attended_values.transpose(1, 2).contiguous().view(
            B, T, C)

        # Apply output projection and dropout
        output = self.resid_dropout(self.output_proj(attended_values))

        return output


class Block(nn.Module):
    """ an unassuming Transformer block """

    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        if config.pretrained:
            self.attn = PretrainedCausalSelfAttention(config)
        else:
            self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = nn.ModuleDict(
            dict(
                c_fc=nn.Linear(config.n_embd, 4 * config.n_embd),
                c_proj=nn.Linear(4 * config.n_embd, config.n_embd),
                act=NewGELU(),
                dropout=nn.Dropout(config.resid_pdrop),
            ))
        m = self.mlp
        self.mlpf = lambda x: m.dropout(m.c_proj(m.act(m.c_fc(x)))
                                        )  # MLP forward

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlpf(self.ln_2(x))
        return x


class GPT(nn.Module):
    """ GPT Language Model """

    @staticmethod
    def get_default_config():
        C = CN()
        # either model_type or (n_layer, n_head, n_embd) must be given in the config
        C.model_type = 'gpt'
        C.n_layer = None
        C.n_head = None
        C.n_embd = None
        # these options must be filled in externally
        C.vocab_size = None
        C.block_size = None
        # dropout hyperparameters
        C.embd_pdrop = 0.1
        C.resid_pdrop = 0.1
        C.attn_pdrop = 0.1
        C.pretrained = False
        return C

    def __init__(self, config):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.block_size = config.block_size

        type_given = config.model_type is not None
        params_given = all([
            config.n_layer is not None, config.n_head is not None,
            config.n_embd is not None
        ])
        assert type_given ^ params_given  # exactly one of these (XOR)
        if type_given:
            # translate from model_type to detailed configuration
            config.merge_from_dict({
                # names follow the huggingface naming conventions
                # GPT-1
                'openai-gpt':
                dict(n_layer=12, n_head=12, n_embd=768),  # 117M params
                # GPT-2 configs
                'gpt2':
                dict(n_layer=12, n_head=12, n_embd=768),  # 124M params
                'gpt2-medium':
                dict(n_layer=24, n_head=16, n_embd=1024),  # 350M params
                'gpt2-large':
                dict(n_layer=36, n_head=20, n_embd=1280),  # 774M params
                'gpt2-xl':
                dict(n_layer=48, n_head=25, n_embd=1600),  # 1558M params
                # Gophers
                'gopher-44m':
                dict(n_layer=8, n_head=16, n_embd=512),
                # (there are a number more...)
                # I made these tiny models up
                'gpt-mini':
                dict(n_layer=6, n_head=6, n_embd=192),
                'gpt-micro':
                dict(n_layer=4, n_head=4, n_embd=128),
                'gpt-nano':
                dict(n_layer=3, n_head=3, n_embd=48),
            }[config.model_type])

        self.transformer = nn.ModuleDict(
            dict(
                wte=nn.Embedding(config.vocab_size, config.n_embd),
                wpe=nn.Embedding(config.block_size, config.n_embd),
                drop=nn.Dropout(config.embd_pdrop),
                h=nn.ModuleList([Block(config)
                                 for _ in range(config.n_layer)]),
                ln_f=nn.LayerNorm(config.n_embd),
            ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # init all weights, and apply a special scaled init to the residual projections, per GPT-2 paper
        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p,
                                      mean=0.0,
                                      std=0.02 / math.sqrt(2 * config.n_layer))

        # report number of parameters (note we don't count the decoder parameters in lm_head)
        n_params = sum(p.numel() for p in self.transformer.parameters())
        print("number of parameters: %.2fM" % (n_params / 1e6, ))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            torch.nn.init.zeros_(module.bias)
            torch.nn.init.ones_(module.weight)

    @classmethod
    def from_pretrained(cls, model_type):
        """
        Initialize a pretrained GPT model by copying over the weights
        from a huggingface/transformers checkpoint.
        """
        assert model_type in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
        from transformers import GPT2LMHeadModel

        # create a from-scratch initialized minGPT model
        config = cls.get_default_config()
        config.model_type = model_type
        config.vocab_size = 50257  # openai's model vocabulary
        config.block_size = 1024  # openai's model block_size
        config.pretrained = True
        model = GPT(config)
        sd = model.state_dict()

        # init a huggingface/transformers model
        model_hf = GPT2LMHeadModel.from_pretrained(model_type)
        sd_hf = model_hf.state_dict()

        # copy while ensuring all of the parameters are aligned and match in names and shapes
        keys = [k for k in sd_hf
                if not k.endswith('attn.masked_bias')]  # ignore these
        keys = [
            k for k in keys
            if not re.match("transformer\.h\.\d+\.attn\.bias", k)
        ]  # ignore these
        sd_keys = [
            k for k in sd if not re.match("transformer\.h\.\d+\.attn\.bias", k)
        ]  # ignore these

        transposed = [
            'attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight',
            'mlp.c_proj.weight'
        ]
        # basically the openai checkpoints use a "Conv1D" module, but we only want to use a vanilla nn.Linear.
        # this means that we have to transpose these weights when we import them

        # This assert might fail for some transformers library versions. Please comment out if that is the case
        assert len(keys) == len(sd_keys)

        for k in keys:
            if any(k.endswith(w) for w in transposed):
                # special treatment for the Conv1D weights we need to transpose
                assert sd_hf[k].shape[::-1] == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k].t())
            else:
                # vanilla copy over the other parameters
                assert sd_hf[k].shape == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k])

        return model

    def configure_optimizers(self, train_config):
        """
        This long function is unfortunately doing something very simple and is being very defensive:
        We are separating out all parameters of the model into two buckets: those that will experience
        weight decay for regularization and those that won't (biases, and layernorm/embedding weights).
        We are then returning the PyTorch optimizer object.
        """

        # separate out all parameters to those that will and won't experience regularizing weight decay
        decay = set()
        no_decay = set()
        whitelist_weight_modules = (torch.nn.Linear, )
        blacklist_weight_modules = (torch.nn.LayerNorm, torch.nn.Embedding)
        for mn, m in self.named_modules():
            for pn, p in m.named_parameters():
                fpn = '%s.%s' % (mn, pn) if mn else pn  # full param name
                # random note: because named_modules and named_parameters are recursive
                # we will see the same tensors p many many times. but doing it this way
                # allows us to know which parent module any tensor p belongs to...
                if pn.endswith('bias'):
                    # all biases will not be decayed
                    no_decay.add(fpn)
                elif pn.endswith('weight') and isinstance(
                        m, whitelist_weight_modules):
                    # weights of whitelist modules will be weight decayed
                    decay.add(fpn)
                elif pn.endswith('weight') and isinstance(
                        m, blacklist_weight_modules):
                    # weights of blacklist modules will NOT be weight decayed
                    no_decay.add(fpn)

        # validate that we considered every parameter
        param_dict = {pn: p for pn, p in self.named_parameters()}
        inter_params = decay & no_decay
        union_params = decay | no_decay
        assert len(
            inter_params
        ) == 0, "parameters %s made it into both decay/no_decay sets!" % (
            str(inter_params), )
        assert len(param_dict.keys() - union_params) == 0, "parameters %s were not separated into either decay/no_decay set!" \
            % (str(param_dict.keys() - union_params), )

        # create the pytorch optimizer object
        optim_groups = [
            {
                "params": [param_dict[pn] for pn in sorted(list(decay))],
                "weight_decay": train_config.weight_decay
            },
            {
                "params": [param_dict[pn] for pn in sorted(list(no_decay))],
                "weight_decay": 0.0
            },
        ]
        optimizer = torch.optim.AdamW(optim_groups,
                                      lr=train_config.learning_rate,
                                      betas=train_config.betas)
        return optimizer

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        assert t <= self.block_size, f"Cannot forward sequence of length {t}, block size is only {self.block_size}"
        pos = torch.arange(0, t, dtype=torch.long,
                           device=device).unsqueeze(0)  # shape (1, t)

        # forward the GPT model itself
        tok_emb = self.transformer.wte(
            idx)  # token embeddings of shape (b, t, n_embd)
        pos_emb = self.transformer.wpe(
            pos)  # position embeddings of shape (1, t, n_embd)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)

        # if we are given some desired targets also calculate the loss
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1),
                                   ignore_index=-1)

        return logits, loss

    @torch.no_grad()
    def generate(self,
                 idx,
                 max_new_tokens,
                 temperature=1.0,
                 do_sample=False,
                 top_k=None):
        """
        Take a conditioning sequence of indices idx (LongTensor of shape (b,t)) and complete
        the sequence max_new_tokens times, feeding the predictions back into the model each time.
        Most likely you'll want to make sure to be in model.eval() mode of operation for this.
        """
        for _ in range(max_new_tokens):
            # if the sequence context is growing too long we must crop it at block_size
            idx_cond = idx if idx.size(
                1) <= self.block_size else idx[:, -self.block_size:]
            # forward the model to get the logits for the index in the sequence
            logits, _ = self(idx_cond)
            # pluck the logits at the final step and scale by desired temperature
            logits = logits[:, -1, :] / temperature
            # optionally crop the logits to only the top k options
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float('Inf')
            # apply softmax to convert logits to (normalized) probabilities
            probs = F.softmax(logits, dim=-1)
            # either sample from the distribution or take the most likely element
            if do_sample:
                idx_next = torch.multinomial(probs, num_samples=1)
            else:
                _, idx_next = torch.topk(probs, k=1, dim=-1)
            # append sampled index to the running sequence and continue
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

    def gen_batch(self, idx, max_new_tokens, temperature=1.0, batch=10):
        """
        A dummy function for "fixed" test generation
        We take a conditioning sequence of indices idx (LongTensor of shape (b,t)),
        take the top "batch" predictions for first token and then complete 10 generations as normal
        Most likely you'll want to make sure to be in model.eval() mode of operation for this.
        """

        out = []
        # if the sequence context is growing too long we must crop it at block_size
        idx_cond = idx if idx.size(
            1) <= self.block_size else idx[:, -self.block_size:]

        # forward the model to get the logits for the index in the sequence
        logits, _ = self(idx_cond)

        # pluck the logits at the final step and scale by desired temperature
        logits = logits[:, -1, :] / temperature

        # apply softmax to convert logits to (normalized) probabilities
        probs = F.softmax(logits, dim=-1)

        # Get the top "batch" predictions for the word
        _, idx_list = torch.topk(probs, k=batch, dim=-1)

        for idx_next in idx_list[0, :]:
            idx_tmp = torch.cat((idx, idx_next.reshape(-1, 1)), dim=1)

            idx_tmp = self.generate(idx_tmp, max_new_tokens - 1)

            out.append(idx_tmp)

        return (out)

    def prompt(self, p_text="", tokens=20, num_samples=1, do_sample=True):
        """
        Human-usable promting function, for the most part just run with prompt and tokens
        """

        if not hasattr(self, 'tok'):
            self.tok = BPETokenizer()

        if p_text == '':
            # to create unconditional samples...
            # manually create a tensor with only the special <|endoftext|> token
            # similar to what openai's code does here https://github.com/openai/gpt-2/blob/master/src/generate_unconditional_samples.py
            x = torch.tensor([[self.tok.encoder.encoder['<|endoftext|>']]],
                             dtype=torch.long)
        else:
            device = next(self.parameters()).device
            x = self.tok(p_text).to(device)

        # we'll process all desired num_samples in a batch, so expand out the batch dim
        x = x.expand(num_samples, -1)

        # forward the model `steps` times to get samples, in a batch
        y = self.generate(x,
                          max_new_tokens=tokens,
                          do_sample=do_sample,
                          top_k=100)

        for i in range(num_samples):
            out = self.tok.decode(y[i].cpu().squeeze())
            print('-' * 80)
            print(out)

    def prompt_topK(self, p_text="", tokens=20, num_samples=5):
        """
        Human-usable prompting function. Deterministic, cah use for evaluation

        """

        if not hasattr(self, 'tok'):
            self.tok = BPETokenizer()

        if p_text == '':
            # to create unconditional samples...
            # manually create a tensor with only the special <|endoftext|> token
            # similar to what openai's code does here https://github.com/openai/gpt-2/blob/master/src/generate_unconditional_samples.py
            x = torch.tensor([[self.tok.encoder.encoder['<|endoftext|>']]],
                             dtype=torch.long)
        else:
            device = next(self.parameters()).device
            x = self.tok(p_text).to(device)

        y = self.gen_batch(x, max_new_tokens=tokens, batch=num_samples)

        for y_tmp in y:
            out = self.tok.decode(y_tmp.cpu().squeeze())
            print('-' * 80)
            print(out)


### Exercise 2.3: Questions [Optional]
<details>
<summary>What is the purpose of applying a causal mask in the attention computation?</summary>
The causal mask ensure that the in attention computation, each position in the sequence can only attend to the positions on its left, preventing information leakage from future positions. This is essential in tasks where the model should generate output sequentially, such as language generation or autoregressive tasks.
</details>

<details>
<summary>How does the number of attention heads affect the model's capacity to capture different types of dependencies in the input sequence?</summary>
Multiple heads allow the model to attend to different parts of the input sequence simultaneously. By increasing the number of attention heads, the model can capture more diverse dependencies and patterns in the data. Each head can focus on different aspects of the input, enabling the model to learn complex relationships and improve performance on tasks that require capturing multiple types of dependencies.
</details>


<details>
<summary>What is the purpose of the residual dropout and attention dropout in the CausalSelfAttention module?</summary>
The residual dropout and attention dropout are regularization techniques used to prevent overfitting and improve the generalization of the model. The residual dropout applies dropout to the output of the attention module, helping to regularize the model during training. The attention dropout applies dropout to the attention weights, which helps to reduce over-reliance on specific tokens and encourages the model to attend to a broader range of tokens in the sequence.
</details>



### Exercise 2.4: Visualize Attentions[Optional]

Now that we understand the basic mechanisms of attention, we can check the activated attention patterns in a pretrained BERT model (Devlin et al. 2018). Recall that BERT is an encoder-based transformer model which is based on a stack of self-attention blocks.

In [29]:
from transformers import BertTokenizer, BertModel
from bertviz import head_view

# Define a sample input text
text = "I will go for a run and will jump into a lake."

# Instantiate the BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# Tokenize the input text
tokens = tokenizer.tokenize(text)

# Convert tokens to token IDs
token_ids = tokenizer.convert_tokens_to_ids(tokens)

# Create attention mask
attention_mask = [1] * len(token_ids)

# Convert token IDs and attention mask to tensors
input_ids = torch.tensor([token_ids])
attention_mask = torch.tensor([attention_mask])

# Generate the transformer output
outputs = model(input_ids, attention_mask=attention_mask, output_attentions=True)

# Extract attentions and check the shape
outputs.attentions[0].shape

ModuleNotFoundError: No module named 'bertviz'

As you can see, we extracted an attention from the first layaer. The first dimension is the bach, the second one is the number of heads used in the first layer, and the last two dimensions are the sequence length. Given that this was a self attention block the last two numbers are equal.

We can now use a method from the [bertviz library](https://github.com/jessevig/bertviz) and plot all the heads.

You'll see a dropdown menu that allows you the select a layer of the model (GPT-2 has 12). You'll then see a color for every head used in that layer (GPT-2 has 12 head per layer). By default all heads are shown, click on a color to activate/disactivate that head. It can help starting by activating only one head and checking the learned relation learn by that self attentino head. By hovering over each word you can see the attention weigths that linked that words to all the others.

**Question** Do you notice any interesting (linguistic) pattern?

In [ ]:
head_view(outputs.attentions, tokens=tokens)

## Exercise 3:
Now that we know everything aboout attention, we can go ahead and train a GPT-based model which heavilty relies on attention.

We will now:
1. Create a GPT-2 model
2. Train this model on a small dataset.
3. Check the loss of our model 

Based on this
https://pytorch.org/tutorials/beginner/transformer_tutorial.html

### Exercise 3.1: Training a Weather Prediction Model using Autoregressive Transformer

In this exercise, we will work with a dummy weather dataset that consists of sequences of weather observations and corresponding states. The goal is to train a small model using an autoregressive transformer to predict the weather state based on the previous observations.

In [1]:
import torch
from torch.utils.data.dataloader import DataLoader
import numpy as np
import time

import random
random.seed(42)

from lxmls.transformers.utils import set_seed
from lxmls.transformers.bpe import BPETokenizer
#from lxmls.transformers.model import GPT
from lxmls.transformers.trainer import Trainer
from lxmls.transformers.dataset import WeatherDataset

We start by initializing the dataset, which is responsible for providing the training data for our model. The dataset contains sequences of weather observations and their corresponding states. These sequences are converted into indices and concatenated to form the input and output sequences for the transformer model.

You can check in detail the dataset in `lxmls/transformers/dataset.py`. 

In [39]:
# Fixed probabilities, easier to learn
# This is just to create the sequence in the dataset
fixed_proba = {}
fixed_proba["initial"] = [.5,.3,.2]
fixed_proba["transition"] = [
    [.5,.5,0],
    [0,.5,.5],
    [.5,0,.5]
]
fixed_proba["emission"] = [
    [.5,0,.2,0,.3],
    [0,.5,.4,0,.1],
    [0,0,.1,.5,.4]
]

In [40]:
# print an example instance of the dataset
train_dataset = WeatherDataset('train', proba=fixed_proba)
test_dataset = WeatherDataset('test', proba=train_dataset.proba)
x, y = train_dataset[0]

print("Sampling from the dataset:")
print(f"Input: {train_dataset.decode_obs(x.tolist()[:6])}")
print(f"Labels: {train_dataset.decode_st(y.tolist()[5:])}")
print("-"*50)
print("Tokenized sequences:")
print(f"Input: {x.tolist()}")
print(f"Labels: {y.tolist()}")

Sampling from the dataset:
Input: ['shop', 'shop', 'clean', 'clean', 'clean', 'read']
Labels: ['rainy', 'rainy', 'rainy', 'rainy', 'rainy', 'snowy']
--------------------------------------------------
Tokenized sequences:
Input: [2, 2, 0, 0, 0, 1, 5, 5, 5, 5, 5]
Labels: [-1, -1, -1, -1, -1, 5, 5, 5, 5, 5, 6]


Next, we create a model using the default configuration for the GPT model. This configuration includes parameters which determine the size and structure of the model. The GPT model is a small version called GPT Nano.

In [46]:
# create a GPT instance
model_config = GPT.get_default_config()
model_config.model_type = 'gpt-nano'
model_config.vocab_size = train_dataset.get_vocab_size()
model_config.block_size = train_dataset.get_block_size()
model = GPT(model_config)

print(model_config)

number of parameters: 0.09M
model_type: gpt-nano
n_layer: 3
n_head: 3
n_embd: 48
vocab_size: 8
block_size: 11
embd_pdrop: 0.1
resid_pdrop: 0.1
attn_pdrop: 0.1
pretrained: False



To train our model, we create a Trainer object. The Trainer handles the training process, including defining the learning rate, setting the maximum number of iterations, and specifying the number of workers for data loading. The Trainer is initialized with the model, training dataset, and validation dataset.

In [47]:
# create a Trainer object
train_config = Trainer.get_default_config()
train_config.learning_rate = 5e-4 # the model we're using is so small that we can go a bit faster
train_config.max_iters = 2000
train_config.num_workers = 0
train_config.device = "cuda"
trainer = Trainer(train_config, model, train_dataset)

print(train_config)

running on device cuda
device: cuda
num_workers: 0
max_iters: 2000
batch_size: 64
learning_rate: 0.0005
betas: (0.9, 0.95)
weight_decay: 0.1
grad_norm_clip: 1.0



With these components in place, we are ready to train our model on the weather dataset and make predictions based on the learned patterns. We just add some minor utilities function that show us intermediate logs. You can safely ignore them since most of this is usually abstracted away from end users in modern deep learning libraries.

In [48]:
def batch_end_callback(trainer):
    if trainer.iter_num % 100 == 0:
        print(f"iter_dt {trainer.iter_dt * 1000:.2f}ms; iter {trainer.iter_num}: train loss {trainer.loss.item():.5f}")
trainer.set_callback('on_batch_end', batch_end_callback)

start_time = time.time()
trainer.run()
end_time = time.time()
elapsed_time = end_time - start_time

# Print the training time
print("Training time: {:.2f} seconds".format(elapsed_time))

iter_dt 0.00ms; iter 0: train loss 2.07355
iter_dt 15.74ms; iter 100: train loss 0.68565
iter_dt 16.13ms; iter 200: train loss 0.38071
iter_dt 15.96ms; iter 300: train loss 0.28526
iter_dt 16.03ms; iter 400: train loss 0.29593
iter_dt 17.02ms; iter 500: train loss 0.28121
iter_dt 16.87ms; iter 600: train loss 0.25028
iter_dt 18.52ms; iter 700: train loss 0.28134
iter_dt 15.50ms; iter 800: train loss 0.30663
iter_dt 15.60ms; iter 900: train loss 0.27203
iter_dt 16.68ms; iter 1000: train loss 0.33331
iter_dt 17.27ms; iter 1100: train loss 0.26686
iter_dt 16.66ms; iter 1200: train loss 0.29461
iter_dt 17.45ms; iter 1300: train loss 0.29458
iter_dt 17.76ms; iter 1400: train loss 0.28477
iter_dt 16.46ms; iter 1500: train loss 0.26326
iter_dt 17.24ms; iter 1600: train loss 0.32218
iter_dt 16.12ms; iter 1700: train loss 0.24295
iter_dt 17.29ms; iter 1800: train loss 0.27687
iter_dt 17.38ms; iter 1900: train loss 0.26853
Training time: 32.55 seconds


As you can see the loss started decreaseing and it seemed to fluctuate around a range of values close to 0.25

Great! You have just **trained a small GPT model**! Congrats!
Generating from such a tiny model that has been trained only for a short number of iteration won't give us interesting output. Let's rely on the one of the many powerful and larger pretrained model publicly available.

### Exercise 3.2: Prompting a pretrained GPT-2 model

We can load a pretrained gpt2 model from hugging face (this is done behind the scene from the GPT class) and prompt it with any text of our choice. 

In [4]:
model_type = "gpt2"
device = "cuda"
model = GPT.from_pretrained(model_type)

# move model to the device(GPU if available)
# set to eval mode to avoid gradient accumulation model.to(device)
model.to(device)
model.eval()

number of parameters: 124.44M


GPT(
  (transformer): ModuleDict(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): PretrainedCausalSelfAttention(
          (c_attn): Linear(in_features=768, out_features=2304, bias=True)
          (c_proj): Linear(in_features=768, out_features=768, bias=True)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModuleDict(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          (act): NewGELU()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head

Great! We can now generate from our pretrained model. We just need to pass a context and let the model generate. 
There are two additional parameters. `tokens` is the number of tokens we want our model to generate and `num_samples` is the number of diverse samples we are asking the model to produce. Since we are sampling from the model distribution, we can generate as many samples as we want, hence the parameter.

Feel free to change the context and have fun _generating text from a pretrained **GPT2**_!

In [5]:
# Random prompt, uses pooling
for i in range(5): 
    set_seed(42)
    model.prompt("Barack Obama, the", 50, 3)

# Deterministic prompt, does NOT use pooling
for i in range(5):
    model.prompt_topK("Barack Obama, the", 50, 3)

--------------------------------------------------------------------------------
Barack Obama, the presumptive GOP nominee, called the Supreme Court's Citizens United decision a "crony capitalism" that has "enormous upside." "This ruling is yet another reason to repeal Obamacare," he said in a statement.

But the decision by
--------------------------------------------------------------------------------
Barack Obama, the GOP nominee, said he would let the federal agency handle "all federal investigations into Benghazi attacks" and asked for stronger oversight on private employers.

Speaking to the conservative Los Angeles Times, Chris Christie agreed that "legislating is our real enemy
--------------------------------------------------------------------------------
Barack Obama, the Democratic presidential front-runner, has suggested it'd be better for the Obama administration to leave the subject to the American people to decide.

"We could find more time to just tell Nancy Pelosi an